In [1]:
import time
import numpy as np
import pandas as pd
from collections import Counter
from IPython.display import display
import sklearn
from sklearn import set_config
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import CategoricalNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    KBinsDiscretizer,
    FunctionTransformer,
    LabelEncoder
)
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

set_config(transform_output="pandas") # For easier debugging with DataFrames and because we work with catecory data type

In [2]:
df = pd.read_parquet('data_processed.parquet')

# pipelines

In [3]:
feature_summary = pd.DataFrame({
    'Feature': df.columns,
    'Unique_Values': [df[col].nunique() for col in df.columns],
    'Data_Type': [df[col].dtype for col in df.columns]
})
feature_summary

,Feature,Unique_Values,Data_Type
0,acrs_report_type,3,category
1,route_type,5,category
2,distance,5862,float64
3,weather,7,category
4,light,6,category
5,driver_substance_abuse,2,float64
6,junction,7,category
7,road_condition,5,category
8,surface_condition_simple,5,category
9,surface_condition_aggressive,4,category


In [4]:
def create_cyclic_features(df):
    """
    Transforms temporal features into sine/cosine components.
    Fixes the TypeError by explicitly casting mapped values to float.
    """
    df = df.copy()
    
    # --- 1. Basic numeric cycles (Hour, Month, Day of week) ---
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'].astype(float) / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'].astype(float) / 24)
    
    df['month_sin'] = np.sin(2 * np.pi * (df['month'].astype(float) - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df['month'].astype(float) - 1) / 12)
    
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'].astype(float) / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'].astype(float) / 7)
    
    # --- 2. Season cycle ---
    # Ensure all possible values in your data are covered by the map
    season_map = {'Winter': 0, 'Spring': 1, 'Summer': 2, 'Autumn': 3}
    if 'season' in df.columns:
        # Crucial fix: .astype(float) after mapping
        season_num = df['season'].map(season_map).astype(float)
        df['season_sin'] = np.sin(2 * np.pi * season_num / 4)
        df['season_cos'] = np.cos(2 * np.pi * season_num / 4)
    
    # --- 3. Time of day cycle ---
    # Double-check if 'Mid_day' should be 'Afternoon' based on your unique values
    time_map = {'Morning': 0, 'Mid_day': 1, 'Evening': 2, 'Night': 3}
    if 'time_of_day' in df.columns:
        # Crucial fix: .astype(float) after mapping
        time_num = df['time_of_day'].map(time_map).astype(float)
        df['time_sin'] = np.sin(2 * np.pi * time_num / 4)
        df['time_cos'] = np.cos(2 * np.pi * time_num / 4)
    
    # Drop original columns to avoid redundancy
    original_cols = ['hour', 'month', 'day_of_week', 'season', 'time_of_day']
    df = df.drop(columns=[c for c in original_cols if c in df.columns], errors='ignore')
    
    return df

In [5]:
# ==========================================
# FEATURE GROUPS SELECTOR
# ==========================================

TARGET_COL = 'acrs_report_type'

# --- Numeric Features ---
# Now includes the generated cyclic sine/cosine features
NUMERIC_FEATS = [
    'distance', 
    'crash_year',
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dow_sin', 'dow_cos',
    'season_sin', 'season_cos',
    'time_sin', 'time_cos'
]

# --- Cyclic Features (Before transformation) ---
# These will be passed through the create_cyclic_features function
CYCLIC_FEATS = ['hour', 'month', 'day_of_week', 'season', 'time_of_day']

# --- Binary Features (0/1) ---
# Note: 'is_two_way' has NaNs in the data and will require imputation
BINARY_FEATS = [
    'is_weekend',
    'is_holiday',
    'is_two_way', # Derivative of road_division_cat
    'driver_substance_abuse',
    # Road Grade Binary Indicators (Handled manually in preprocessing)
    'road_grade_Downhill', 'road_grade_Hillcrest', 'road_grade_Level', 
    'road_grade_On_Bridge', 'road_grade_Other', 'road_grade_Sag', 
    'road_grade_Uphill', 'road_grade_Not_Relevant',
    # Road Alignment Binary Indicators
    'road_alignment_Curve_Left', 'road_alignment_Curve_Right', 
    'road_alignment_Other', 'road_alignment_Straight', 
    'road_alignment_Not_Relevant'
]

# --- Nominal Categorical Features ---
# Features to be One-Hot Encoded. 
# Note: 'road_grade_cat' and 'road_alignment_cat' are removed to avoid redundancy 
# with the binary indicators listed above.
CAT_FEATS = [
    'route_type', 
    'weather', 
    'junction', 
    'road_condition',
    'light', 
    'surface_condition_simple', 
    'surface_condition_aggressive',
    'road_grade_cat',    
    'road_alignment_cat',
    'road_division_cat', 
    'division_type', # Derivative of road_division_cat
    'is_divided',  # Derivative of road_division_cat
    'has_left_turn', # Derivative of road_division_cat
    'traffic_control_cat'
]

# Final consolidated list for X matrix construction
ALL_FEATURES = NUMERIC_FEATS + BINARY_FEATS + CAT_FEATS

--- Redundancy Check ---  

 Note: If using a model like Logistic Regression, avoid using both 'road_grade_cat'   
 
 and the binary 'road_grade_...' indicators simultaneously to prevent multi-collinearity.

In [12]:
# ==============================
# TREE-MODELS (CB, LGBM, RF)
# ===============================

# ============================================================
# FEATURE SELECTION 
# Strategy: Use parent categorical columns instead of binary 
# derivatives to prevent information dilution and redundancy.
# ============================================================

# 1. Categorical: Using rich "parent" columns for optimal splitting.
# We prefer 'simple' surface condition and exclude the 'aggressive' version.
tree_cat_selection = [
    'route_type', 'weather', 'junction', 'road_condition', 'light',
    'surface_condition_simple', 
    'road_grade_cat', 
    'road_alignment_cat', 
    'road_division_cat', 
    'traffic_control_cat'
]

# 2. Binary: Removing derivatives of Grade, Alignment, and Division.
# Trees handle these better when kept together in their categorical form.
tree_bin_selection = [c for c in BINARY_FEATS if not (
    c.startswith('road_grade_') or 
    c.startswith('road_alignment_') or 
    c == 'is_two_way')] # Derivative of road_division_cat

native_preprocessor = ColumnTransformer(
    transformers=[
        # Numeric/Binary: keep NaNs; CatBoost handles missing values natively
        # Categorical: encode missingness explicitly as "MISSING"
        ('num', 'passthrough', NUMERIC_FEATS),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
            ('cast', FunctionTransformer(lambda x: x.astype('category'), validate=False))
        ]), tree_cat_selection), 
        ('bin', 'passthrough', tree_bin_selection)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)


# --- CatBoost ---
cb_pipeline = Pipeline([
     ('preprocessor', native_preprocessor),
    ('model', CatBoostClassifier(
        # categorical features by index after ColumnTransformer ordering
        # cat_features = list(
        #     range(len(NUMERIC_FEATS),
        #           len(NUMERIC_FEATS) + len(tree_cat_selection))), 
        cat_features = tree_cat_selection,
        auto_class_weights='Balanced', # Increases penalty for mistakes on rare classes
        verbose=50,
        allow_writing_files=False,
        loss_function='MultiClass', # penalizes the model for being "confident and wrong" + softmax
        random_state=42
    ))
])

# --- LightGBM ---
lgbm_pipeline = Pipeline([
    ('preprocessor', native_preprocessor),
    ('model', LGBMClassifier(
        class_weight='balanced', # Increases penalty for mistakes on rare classes
        verbose=10,
        n_jobs=-1,
        categorical_feature='auto',
        objective='multiclass', # multi-class classification, softmax
        num_class=3, 
        metric='multi_logloss', # penalizes the model for being "confident and wrong"
        importance_type='gain', # Better for interpretation and more fit to our goal
        random_state=42
    ))
])

# --- Random Forest ---
rf_preprocessor = ColumnTransformer(
    transformers=[
        # Numeric: Simple median imputation
        ('num', SimpleImputer(strategy='median'), NUMERIC_FEATS), 
        
        # Binary: Imputing missing values (RF fails on NaNs in sklearn)
        ('bin', SimpleImputer(strategy='most_frequent'), tree_bin_selection),

        # Categorical: All tree categories via OHE
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
        ]), tree_cat_selection)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

rf_pipeline = Pipeline([
    ('preprocessor', rf_preprocessor),
    ('model', RandomForestClassifier(
        class_weight='balanced',  # importent for rare classes

        n_estimators=200,      
        max_depth=15,          # Limit depth to prevent Overfitting on noisy data
        min_samples_split=10,  # Require at least 10 samples to split a node
        min_samples_leaf=4,    # helps smooth the model against noise
        n_jobs=-1,         
        random_state=42,
        verbose=0           
    ))
])

In [7]:
# ======================================================
# PREPROCESSOR FOR LOGISTIC REGRESSION
# ======================================================
# Categorical: Only those that DON'T have binary equivalents in lBINARY_FEATS.
# We exclude the 'cat' versions of road structure and the 'aggressive' surface.
lr_cat_selection = [c for c in CAT_FEATS if c not in [
    'road_grade_cat', 
    'road_alignment_cat', 
    'road_division_cat', 
    'surface_condition_aggressive'
]]


lr_preprocessor = ColumnTransformer(
    transformers=[
        # Numeric: scale + impute
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()) 
        ]), NUMERIC_FEATS),

        ('bin', SimpleImputer(strategy='most_frequent'), BINARY_FEATS),

        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), lr_cat_selection)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

# ======================================================
# MODEL PIPELINE: LOGISTIC REGRESSION
# ======================================================
lr_pipeline = Pipeline([
    ('preprocessor', lr_preprocessor),
    ('model', LogisticRegression(
        # --- Solver & Convergence ---
        solver='lbfgs',          # The standard engine (Stable, handles Multiclass, supports L2, suitable for our data size)
        max_iter=500,            # Changed from default (100) to ensure convergence on imbalanced data
        
        # --- Class Imbalance ---
        class_weight='balanced', # Forces the model to give more weight to rare classes, maybe we will manually weight later
        
        # --- Technical ---
        n_jobs=-1,             
        random_state=42          
        
        # --- Hidden Defaults ---
        # penalty='l2' (Standard Ridge Regularization is on)
        # C=1.0 (Moderate regularization strength)
        # multi_class='auto' (Automatically selects Softmax/Multinomial)
    ))
])

In [8]:
# ======================================================
# PREPROCESSOR FOR NAIVE BAYES
# ======================================================

# NB is extremely sensitive to correlated features (double-counting). 
# also he asumes indrependent features so we prfer the "father" features and not their derivatives.
# We use the clean "Tree" selection.

nb_cat_selection = tree_cat_selection + ['crash_year']  # Moving discrete numeric to categorical for better NB performance
nb_num_feats = [c for c in NUMERIC_FEATS if c != 'crash_year']


nb_preprocessor = ColumnTransformer(
    transformers=[
        # --- Discretizing Numeric Features ---
        # CategoricalNB requires discrete categories. 
        # Quantile strategy ensures bins have equal number of samples.
        ('num_binned', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('discretizer', KBinsDiscretizer(
                n_bins=10, 
                encode='ordinal', 
                strategy='quantile'
            ))
        ]), nb_num_feats),
        
        # --- Binary Features ---
        # Already discrete (0/1), just need to ensure no NaNs.
        ('bin', SimpleImputer(strategy='most_frequent'), tree_bin_selection),
        
        # --- Categorical Features ---
        # Shifted to ensure all values are non-negative integers.
        ('cat_all', Pipeline([
            ('to_str', FunctionTransformer(lambda x: x.astype(str), validate=False)),
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('ordinal', OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1 
            )),
            ('shift', FunctionTransformer(func=lambda x: x + 1, validate=False
            ))
        ]), nb_cat_selection)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

# ======================================================
# MODEL PIPELINE: CATEGORICAL NAIVE BAYES
# ======================================================
nb_pipeline = ImbPipeline([
    ('preprocessor', nb_preprocessor),
    
    # NB does not have a 'class_weight' parameter in sklearn.
    # Resampling is necessary to handle the minority classes.
    ('resampler', RandomOverSampler(
        sampling_strategy='not majority',  # maybe we will try dictionary later
        random_state=42
    )),
    
    ('model', CategoricalNB(
        alpha=1.0  # Laplace smoothing to avoid zero probabilities. high -> more wheigt to prior. low -> more weight to data.
    ))
])

## הרצה והשוואה

In [9]:
y = df[TARGET_COL]
X_raw = df.drop(columns=[TARGET_COL])

X = create_cyclic_features(X_raw) # creates the hour_sin, month_cos etc. features we defined

# --- Stratified Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Target Encoding ---
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

target_names = list(le.classes_)


counts = Counter(y_train_encoded)
total = len(y_train_encoded)

# Building the summary table
summary_data = []
for i, class_name in enumerate(le.classes_):
    count = counts[i]
    summary_data.append({
        'Label': i,
        'Class Name': class_name,
        'Count': f"{count:,}",
        'Percentage': f"{(count/total)*100:.2f}%"
    })

# Identifying the minority class (index and name) for the benchmark
minority_class_val = min(counts, key=counts.get)
minority_class_name = le.classes_[minority_class_val]

print("📊 Target Variable Summary (Train Set):")
print("-" * 45)
print(pd.DataFrame(summary_data).to_string(index=False))
print("-" * 45)
print(f"Target Minority Class: {minority_class_val} ({minority_class_name})\n")

📊 Target Variable Summary (Train Set):
---------------------------------------------
 Label            Class Name  Count Percentage
     0           Fatal Crash    295      0.32%
     1          Injury Crash 31,601     33.81%
     2 Property Damage Crash 61,577     65.88%
---------------------------------------------
Target Minority Class: 0 (Fatal Crash)



In [13]:
# ======================================================
# 3. BENCHMARK FUNCTION
# ======================================================
def run_benchmark(pipelines_dict, X_train, y_train, X_test, y_test, minority_class_idx, target_names):
    """
    Evaluates multiple pipelines and returns a performance leaderboard.
    """
    results = []
    
    for name, pipe in pipelines_dict.items():
        print(f"🚀 Processing: {name}...")
        
        # Training
        start_time = time.time()
        pipe.fit(X_train, y_train)
        train_time = time.time() - start_time
        
        # Prediction
        y_pred = pipe.predict(X_test)
        
        # Global Metrics
        f1_macro = f1_score(y_test, y_pred, average='macro')
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        
        # Per-class Metrics
        # Using labels=range(len(target_names)) to match our 0, 1, 2 encoding
        prec, rec, f1_per_class, _ = precision_recall_fscore_support(
            y_test, y_pred, labels=range(len(target_names)), zero_division=0
        )
        
        # Store results for the specific minority class index
        results.append({
            'Model': name,
            'F1 Macro': round(f1_macro, 4),
            'Balanced Acc': round(bal_acc, 4),
            'Min. Recall': round(rec[minority_class_idx], 4),
            'Min. Precision': round(prec[minority_class_idx], 4),
            'Min. F1': round(f1_per_class[minority_class_idx], 4),
            'Time (s)': round(train_time, 2)
        })
        
        # Intermediate output for tracking with CATEGORY NAMES
        print(f"\n📊 Classification Report for {name}:")
        print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0, digits=3))
        print("-" * 60)

    # Leaderboard creation
    leaderboard = pd.DataFrame(results).sort_values('F1 Macro', ascending=False)
    return leaderboard

# ======================================================
# 4. EXECUTION
# ======================================================
pipelines = {
    'LR': lr_pipeline,
    'NB': nb_pipeline,
    'RF': rf_pipeline,
    'CB': cb_pipeline,  
    'LGBM': lgbm_pipeline 
}

# IMPORTANT: Using the encoded variables and names from your first block
leaderboard = run_benchmark(
    pipelines, 
    X_train, 
    y_train_encoded, 
    X_test, 
    y_test_encoded, 
    minority_class_val, # The index (0, 1, or 2)
    target_names        # The list of strings ('Minor', etc.)
)

print("\n🏆 FINAL LEADERBOARD")
print("=" * 70)
print(leaderboard.to_string(index=False))

🚀 Processing: LR...

📊 Classification Report for LR:
                       precision    recall  f1-score   support

          Fatal Crash      0.009     0.716     0.018        74
         Injury Crash      0.417     0.475     0.444      7900
Property Damage Crash      0.767     0.421     0.543     15395

             accuracy                          0.440     23369
            macro avg      0.398     0.537     0.335     23369
         weighted avg      0.646     0.440     0.508     23369

------------------------------------------------------------
🚀 Processing: NB...


C:\Users\אליאור\AppData\Roaming\Python\Python311\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\אליאור\AppData\Roaming\Python\Python311\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\אליאור\AppData\Roaming\Python\Python311\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 2 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\אליאור\AppData\Roaming\Python\Python311\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 3 are removed. Consider decreasing the number of bins.
  warning


📊 Classification Report for NB:
                       precision    recall  f1-score   support

          Fatal Crash      0.008     0.635     0.015        74
         Injury Crash      0.393     0.621     0.481      7900
Property Damage Crash      0.815     0.244     0.376     15395

             accuracy                          0.373     23369
            macro avg      0.405     0.500     0.291     23369
         weighted avg      0.670     0.373     0.410     23369

------------------------------------------------------------
🚀 Processing: RF...

📊 Classification Report for RF:
                       precision    recall  f1-score   support

          Fatal Crash      0.015     0.027     0.019        74
         Injury Crash      0.420     0.644     0.509      7900
Property Damage Crash      0.749     0.541     0.628     15395

             accuracy                          0.574     23369
            macro avg      0.395     0.404     0.385     23369
         weighted avg      0.

C:\Users\אליאור\AppData\Roaming\Python\Python311\site-packages\lightgbm\basic.py:2137: UserWarning: categorical_feature keyword has been found in `params` and will be ignored.
Please use categorical_feature argument of the Dataset constructor to pass this parameter.
  _log_warning(
C:\Users\אליאור\AppData\Roaming\Python\Python311\site-packages\lightgbm\basic.py:2159: UserWarning: categorical_feature in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


[LightGBM] [Warning] categorical_feature is set=auto, categorical_column=12,13,14,15,16,17,18,19,20,21 will be ignored. Current value: categorical_feature=auto
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.798764
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.263728
[LightGBM] [Debug] init for col-wise cost 0.015753 seconds, init for row-wise cost 0.052093 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028254 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Sparse Multi-Val Bin
[LightGBM] [Info] Total Bins 441
[LightGBM] [Info] Number of data points in the train set: 93473, number of used features: 25
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Debug] Trai